# Vision Transformer Frequency Analysis
## Replicating 'What do Deep Networks Like to See?' using FFT

**Approach:** Input Image → FFT → Learnable Frequency Mask → IFFT → Frozen Classifier

---

## Section 1: Setup & Environment Check

### 1.1 GPU Check

In [ ]:
# Check GPU availability
!nvidia-smi

### 1.2 Install Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision timm matplotlib numpy pillow -q

### 1.3 Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import timm
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import os
import sys
from tqdm import tqdm

print("All libraries imported successfully!")

### 1.4 Verify PyTorch GPU Access

In [ ]:
# Check CUDA availability and device info
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    device = torch.device('cuda')
    print(f"\n✓ Using device: {device}")
    
    # Test GPU with a simple tensor operation
    test_tensor = torch.randn(1000, 1000).to(device)
    result = torch.matmul(test_tensor, test_tensor)
    print(f"✓ GPU test passed - tensor shape: {result.shape}")
else:
    device = torch.device('cpu')
    print(f"\n⚠ WARNING: CUDA not available. Using CPU.")
    print("Please enable GPU in Colab: Runtime → Change runtime type → GPU")

### 1.5 Load ImageNet Validation Dataset

In [ ]:
# ImageNet validation set: 50K images, 1000 classes, 224x224 resolution

# Define transforms for ImageNet (standard preprocessing)
transform_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# OPTION 1: Load ImageNet from local directory (if already downloaded)
# Download from Kaggle: https://www.kaggle.com/c/imagenet-object-localization-challenge
# Upload to Colab and specify path below
imagenet_path = './data/imagenet'  # Update this path if needed

# OPTION 2: Load ImageNet using torchvision (requires authentication)
# Uncomment below if you have ImageNet credentials
# valset = torchvision.datasets.ImageNet(
#     root='./data/imagenet', split='val', transform=transform_val
# )

# For Colab users: Upload ImageNet validation set to Google Drive or use Kaggle API
# Example using manual path:
try:
    valset = torchvision.datasets.ImageFolder(
        root=f'{imagenet_path}/val',  # Assumes structure: imagenet/val/n01440764/...
        transform=transform_val
    )
    valloader = DataLoader(valset, batch_size=64, shuffle=False, num_workers=2)
    
    print(f"\n✓ ImageNet validation set loaded successfully!")
    print(f"  Validation samples: {len(valset)}")
    print(f"  Number of classes: {len(valset.classes)}")
    print(f"  Image resolution: 224x224")
    print(f"  Batch size: 64")
    
except FileNotFoundError:
    print("\n⚠ ImageNet validation set not found!")
    print("\nTo download ImageNet validation set:")
    print("1. From Kaggle: https://www.kaggle.com/c/imagenet-object-localization-challenge")
    print("2. Or use Kaggle API:")
    print("   !pip install kaggle")
    print("   !kaggle competitions download -c imagenet-object-localization-challenge")
    print("   !unzip imagenet-object-localization-challenge.zip -d ./data/imagenet")
    print("\n3. Update 'imagenet_path' variable above to point to your data directory")
    print("\nFor testing purposes, you can start with a subset of ImageNet or use CIFAR-10.")
    
    valset = None
    valloader = None

### 1.6 Visualize Sample Images

In [ ]:
# Function to unnormalize ImageNet images and display
def imshow(img, title=None):
    # Unnormalize using ImageNet mean and std
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img * std + mean
    img = torch.clamp(img, 0, 1)  # Clamp to [0, 1] range
    
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title, fontsize=8)
    plt.axis('off')

# Visualize sample images if dataset loaded successfully
if valloader is not None:
    # Get a batch of validation images
    dataiter = iter(valloader)
    images, labels = next(dataiter)
    
    # Show first 8 images
    plt.figure(figsize=(15, 4))
    for i in range(8):
        plt.subplot(2, 4, i+1)
        imshow(images[i], f"Class: {labels[i].item()}")
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Sample batch shape: {images.shape}")
    print(f"  [batch_size, channels, height, width] = {list(images.shape)}")
    print(f"  Label range: {labels.min().item()} to {labels.max().item()}")
else:
    print("\n⚠ Skipping visualization - dataset not loaded")
    print("Please load ImageNet validation set first (see cell above)")

---

## ✅ Section 1 Complete!

**Environment Check Summary:**
- GPU detected and verified
- All dependencies installed
- Libraries imported
- PyTorch GPU access confirmed
- ImageNet validation set configured (50K images, 1000 classes, 224x224)

**Dataset Info:**
- **ImageNet validation**: 50,000 images across 1,000 classes
- **Resolution**: 224×224 (standard for pre-trained models)
- **Normalization**: ImageNet mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]

**Next Steps:** Ready to implement Section 2 (FFT & Learnable Frequency Mask architecture)